# Toxicity events — Pass 2 (TOB spread/vol join)


In [1]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

BOOK = Path(os.environ.get('ARES_MICROSTRUCTURE') or (Path.home() / 'srv' / 'ares-microstructure')) / 'research' / 'books' / 'vpin_of'
OUT = BOOK / 'out'
FIGS = OUT / 'desk_synthesis' / 'figs'

def jload(rel):
    p = OUT / rel
    return json.loads(p.read_text()) if p.is_file() else {}

def jlines(rel):
    p = OUT / rel
    if not p.is_file():
        return []
    return [json.loads(ln) for ln in p.read_text().splitlines() if ln.strip()]

def show_fig(name, caption=None):
    p = FIGS / name
    if caption:
        display(Markdown(f'**{caption}** — `{p.relative_to(BOOK)}`'))
    if p.exists():
        display(Image(filename=str(p)))
    else:
        display(Markdown(f'*missing* `{p}` — run `scripts/build_notebook_figs.py`'))

dec = jload('pass2/decisions_pass2.json') or jload('vpin_panel/decisions.json')
dec_promote = jload('vpin_panel/decisions_panel_promote.json')
rows = jlines('vpin_panel/rows.jsonl')
panel_ext = jload('pass2/panel_hl_db_extended.json')
ok_rows = panel_ext if isinstance(panel_ext, list) and panel_ext else [r for r in rows if r.get('ok')]
p2 = jload('pass2/pass2_summary.json')
print('pass2' if p2 else 'pass1', 'n_ok', dec.get('n_ok'), 'promote n_ok', dec_promote.get('n_ok'), 'counts', dec.get('decision_counts'))


pass2 n_ok 226 promote n_ok 139 counts {'Promote': 6, 'Hold': 5}


In [2]:
tox = jload('pass2/toxicity.json')
if tox:
    display(Markdown(f"**{tox.get('decision')}** — n_days={tox.get('n_days')} ρ_vpin,spread={((tox.get('spearman_vpin_spread') or {}).get('rho'))}"))
    display(pd.DataFrame(tox.get('sample_top_days', [])))
else:
    d = pd.DataFrame(ok_rows)
    q90 = d['mean_vpin'].quantile(0.9) if not d.empty else float('nan')
    display(Markdown(f'Top decile tape preview (mean_vpin ≥ {q90:.3f}) — run `scripts/exp_pass2.py` for TOB join'))


**Hold** — n_days=73 ρ_vpin,spread=-0.36403184005923733

,ok,venue,symbol,day,mean_vpin,spread_bps_mean,spread_bps_p90,rv_log_mid,tob_source
0,True,deribit,ETH,2026-08-29,0.947137,4.396776,8.750705,0.048854,warehouse:l2_snapshot_level
1,True,hyperliquid,BTC,2026-09-26,0.943741,0.120751,0.119199,0.006586,warehouse:l2_rebuild
2,True,deribit,BTC,2026-09-05,0.941174,6.606091,12.058151,0.086671,warehouse:l2_snapshot_level
3,True,hyperliquid,BTC,2026-09-30,0.939231,0.134091,0.120275,0.015469,collector
4,True,hyperliquid,BTC,2026-09-17,0.936370,0.143857,0.131123,0.011922,warehouse:l2_rebuild
5,True,hyperliquid,BTC,2026-09-25,0.936288,0.130667,0.119409,0.016691,warehouse:l2_rebuild
6,True,deribit,BTC,2026-10-01,0.928932,6.265456,10.761049,0.035161,warehouse:l2_snapshot_level
7,True,deribit,BTC,2026-09-30,0.928874,3.599256,8.116363,0.003465,warehouse:l2_snapshot_level
